In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
from PIL import Image
import os

# 1. 모델 아키텍처 재구성 (위에서 정의된 VGG16 기반 모델과 동일하게)
#    `base_model`과 동일한 파라미터로 다시 생성해야 합니다.
base_model_reloaded = keras.applications.VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(*IMG_SIZE, 3)
)
base_model_reloaded.trainable = False # 반드시 동결 상태로 유지

reloaded_model = keras.Sequential([
    base_model_reloaded,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation='sigmoid', name='predictions')
])

# 2. 저장된 가중치 로드
weights_path = "weights/leather_model.weights.h5"
reloaded_model.load_weights(weights_path)

print(f"가중치로부터 모델 로드 완료: {weights_path}")
reloaded_model.summary()

# 3. 예시 추론 (테스트 데이터셋의 첫 번째 이미지 사용)
if len(test_dataset.image_paths) > 0:
    sample_image_path = test_dataset.image_paths[0]
    sample_pil_image = Image.open(sample_image_path).convert("RGB").resize(IMG_SIZE)

    # 모델 입력용 전처리 (VGG16 전용)
    sample_np_image = np.array(sample_pil_image, dtype=np.float32).copy()
    processed_image = keras.applications.vgg16.preprocess_input(sample_np_image)
    processed_image = np.expand_dims(processed_image, axis=0) # 배치 차원 추가

    # 추론
    prediction_prob = reloaded_model.predict(processed_image, verbose=0)[0][0]
    predicted_label = "불량" if prediction_prob > 0.5 else "정상"

    print(f"\n예시 이미지: {sample_image_path}")
    print(f"예측 확률: {prediction_prob:.4f}")
    print(f"예측 결과: {predicted_label}")
else:
    print("\n테스트 데이터셋에 이미지가 없습니다. 추론 예시를 건너뜝니다.")